# Two-Stage Ekman Emotion Classification

**Pipeline:**
```
Text → [Stage 1: neutral vs emotion] → emotion → [Stage 2: 6-class multi-label] → anger/joy/...
                                      → neutral ─────────────────────────────── → neutral
```

| | Stage 1 | Stage 2 |
|---|---|---|
| **Task** | Binary: neutral vs has-emotion | Multi-label: 6 Ekman emotions |
| **Recommended** | ELECTRA | DeBERTa-v3 |
| **Loss** | BCE + pos_weight | Three-tier Per-Class ASL |
| **Imbalance** | mild (2.5:1) → pos_weight only | severe (10:1 joy/fear) → 3-tier strategy |

**Output structure:**
```
<run_base_dir>/run_2_stage/
  electra+deberta/           ← first run
    checkpoints/             ← stage1_best.pth, stage2_best.pth
    logs/                    ← training_log_stage1.csv, stage2.csv
    results/stage1/          ← report, metrics, charts
    results/stage2/
    results/end2end/
  electra+deberta_2/         ← second run (no overwrite)
```

## 0. Setup

In [ ]:
# Google Drive mount (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted')
except ImportError:
    print('Not on Colab — skipping drive mount')

In [ ]:
!pip install torch transformers scikit-learn pandas numpy matplotlib tqdm pyyaml -q

In [ ]:
import os, sys
import torch

PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
from src.utils import load_config

CONFIG_PATH = 'config/config.yaml'
cfg = load_config(CONFIG_PATH)

print(f"run_base_dir : {cfg['run_base_dir']}")
print(f"Stage 1 model: {cfg['stage1']['model']['name']}")
print(f"Stage 2 model: {cfg['stage2']['model']['name']}")
print(f"Stage 1 loss : {cfg['stage1']['training']['loss']}")
print(f"Stage 2 loss : {cfg['stage2']['training']['loss']}")

## 1. Data Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(os.path.join(cfg['data']['data_dir'], cfg['data']['train_file']))
emotions = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']
counts   = df[emotions].sum().sort_values(ascending=False)

n_emotion = (df[emotions].sum(axis=1) > 0).sum()
n_neutral = len(df) - n_emotion
median    = counts.median()

print(f'Total samples : {len(df):,}')
print(f'has_emotion   : {n_emotion:,} ({n_emotion/len(df)*100:.1f}%)')
print(f'neutral       : {n_neutral:,} ({n_neutral/len(df)*100:.1f}%)')
print(f'\nEmotion counts (sorted):')
for name, c in counts.items():
    tier = 'very_rare' if c < median/3 else 'rare' if c < median else 'common'
    print(f'  {name:<12}: {int(c):>6}  [{tier}]')
print(f'\nmedian={median:.0f}  joy/fear ratio={counts["joy"]/counts["fear"]:.1f}×')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Stage 1 pie
axes[0].pie([n_emotion, n_neutral], labels=['has_emotion', 'neutral'],
            colors=['#e74c3c', '#3498db'], autopct='%1.1f%%', startangle=90)
axes[0].set_title('Stage 1: Binary Balance')

# Stage 2 bar — color by tier
colors = ['#c0392b' if c < median/3 else '#e67e22' if c < median else '#27ae60'
          for c in counts.values]
axes[1].bar(counts.index, counts.values, color=colors, edgecolor='white')
axes[1].axhline(median,   color='black',  linestyle='--', label=f'median={median:.0f}')
axes[1].axhline(median/3, color='gray',   linestyle=':',  label=f'median/3={median/3:.0f}')
axes[1].set_title('Stage 2: Emotion Counts (red=very_rare, orange=rare, green=common)')
axes[1].set_ylabel('Count')
axes[1].legend()
for i, (name, c) in enumerate(counts.items()):
    axes[1].text(i, c + 100, f'{int(c):,}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

## 2. Stage 1 Training

**Task:** Binary — detect if a text has any emotion  
**Imbalance:** mild 2.5:1 → BCE + pos_weight is sufficient  
**Recommended model:** ELECTRA (pre-trained as binary discriminator)

In [ ]:
# ── Optional overrides ────────────────────────────────────────────────────────
# cfg['stage1']['model']['name'] = 'electra'   # electra | bert | roberta | deberta
# cfg['stage1']['training']['epochs'] = 10
# cfg['stage1']['training']['lr'] = 3e-5

from src.train import train

s1_result = train(config_path=CONFIG_PATH, stage='stage1')

RUN_DIR = s1_result['run_dir']   # reuse same run dir for stage2 + test
print(f'\nRun directory: {RUN_DIR}')
print(f'Best epoch   : {s1_result["best_epoch"]}')
for k, v in s1_result['best_metrics'].items():
    print(f'  {k}: {v:.4f}')

In [ ]:
log1 = pd.read_csv(s1_result['log_path'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(log1.epoch, log1.train_loss, label='train', marker='o', ms=4)
ax1.plot(log1.epoch, log1.val_loss,   label='val',   marker='s', ms=4)
ax1.set_title('Stage 1 — Loss')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(log1.epoch, log1.val_f1,        label='F1',        marker='o', ms=4)
ax2.plot(log1.epoch, log1.val_accuracy,  label='Accuracy',  marker='s', ms=4)
ax2.plot(log1.epoch, log1.val_precision, label='Precision', marker='^', ms=4)
ax2.plot(log1.epoch, log1.val_recall,    label='Recall',    marker='v', ms=4)
ax2.set_title('Stage 1 — Val Metrics')
ax2.set_xlabel('Epoch')
ax2.set_ylim(0, 1.05)
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Stage 2 Training

**Task:** Multi-label 6 Ekman emotions (emotion-only samples)  
**Imbalance:** severe (joy/fear ≈ 10×) → 3-tier strategy  
**Recommended model:** DeBERTa-v3  

**Three-tier treatment:**
| Tier | Classes | Aug copies | Sampler boost | pos_weight scale | ASL γ- |
|------|---------|-----------|--------------|-----------------|--------|
| very_rare | fear | ×4 | ×5 | ^2.0 | 0.5 |
| rare | disgust | ×2 | ×3 | ^1.5 | 1.0 |
| common | anger, joy, sadness, surprise | ×0 | ×1 | ^1.0 | 2.0 |

In [ ]:
# ── Optional overrides ────────────────────────────────────────────────────────
# cfg['stage2']['model']['name'] = 'deberta'   # deberta | roberta | electra | bert
# cfg['stage2']['training']['batch_size'] = 16   # reduce if OOM
# cfg['stage2']['training']['epochs'] = 20

s2_result = train(config_path=CONFIG_PATH, stage='stage2', run_dir=RUN_DIR)

print(f'\nBest epoch  : {s2_result["best_epoch"]}')
for k, v in s2_result['best_metrics'].items():
    print(f'  {k}: {v:.4f}')

In [ ]:
log2 = pd.read_csv(s2_result['log_path'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(log2.epoch, log2.train_loss, label='train', marker='o', ms=4)
ax1.plot(log2.epoch, log2.val_loss,   label='val',   marker='s', ms=4)
ax1.set_title('Stage 2 — Loss')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(log2.epoch, log2.val_micro_f1,    label='micro_F1',    marker='o', ms=4)
ax2.plot(log2.epoch, log2.val_macro_f1,    label='macro_F1',    marker='s', ms=4)
ax2.plot(log2.epoch, log2.val_weighted_f1, label='weighted_F1', marker='^', ms=4)
ax2.set_title('Stage 2 — Val F1')
ax2.set_xlabel('Epoch')
ax2.set_ylim(0, 1.05)
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Evaluate Stage 1

In [ ]:
from src.test import evaluate_stage1

s1_metrics = evaluate_stage1(config_path=CONFIG_PATH, run_dir=RUN_DIR)

print(f'Accuracy     : {s1_metrics["accuracy"]:.4f}')
print(f'F1 (emotion) : {s1_metrics["f1"]:.4f}')
print(f'Macro F1     : {s1_metrics["macro_f1"]:.4f}')
print(f'Best threshold: {s1_metrics["best_threshold"]:.2f}')

## 5. Evaluate Stage 2

In [ ]:
from src.test import evaluate_stage2

s2_metrics = evaluate_stage2(config_path=CONFIG_PATH, run_dir=RUN_DIR)

print(f'Micro  F1    : {s2_metrics["micro_f1"]:.4f}')
print(f'Macro  F1    : {s2_metrics["macro_f1"]:.4f}')
print(f'Weighted F1  : {s2_metrics["weighted_f1"]:.4f}')
print(f'Subset Acc   : {s2_metrics["subset_accuracy"]:.4f}')

In [ ]:
# Per-class breakdown
pc = pd.read_csv(os.path.join(s2_metrics['out_dir'], 'stage2_per_class.csv'))
pc = pc.set_index('emotion').sort_values('f1')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['f1', 'precision', 'recall']):
    colors = ['#c0392b' if v < 0.5 else '#e67e22' if v < 0.7 else '#27ae60' for v in pc[col]]
    bars = ax.barh(pc.index, pc[col], color=colors, edgecolor='white')
    ax.set_xlim(0, 1.1)
    ax.set_title(f'Per-class {col.upper()}')
    ax.axvline(pc[col].mean(), color='black', linestyle='--', alpha=0.7,
               label=f'Mean={pc[col].mean():.3f}')
    ax.legend(fontsize=8)
    for bar, v in zip(bars, pc[col]):
        ax.text(v+0.01, bar.get_y()+bar.get_height()/2, f'{v:.2f}', va='center', fontsize=8)

plt.suptitle('Stage 2 — Per-Class Metrics', fontsize=12)
plt.tight_layout()
plt.show()
display(pc)

## 6. End-to-End Evaluation (7 classes)

In [ ]:
from src.test import evaluate_end_to_end

e2e = evaluate_end_to_end(config_path=CONFIG_PATH, run_dir=RUN_DIR)

print(f'Micro  F1    : {e2e["micro_f1"]:.4f}')
print(f'Macro  F1    : {e2e["macro_f1"]:.4f}')
print(f'Weighted F1  : {e2e["weighted_f1"]:.4f}')
print(f'Subset Acc   : {e2e["subset_accuracy"]:.4f}')
print(f'Stage1 Acc   : {e2e["stage1_accuracy"]:.4f} (routing quality)')

In [ ]:
e2e_pc = pd.read_csv(os.path.join(e2e['out_dir'], 'e2e_per_class.csv'))
e2e_pc = e2e_pc.set_index('class')

from src.dataloader import ALL_CLASS_NAMES
colors = ['#3498db' if c == 'neutral' else
          '#c0392b' if e2e_pc.loc[c,'f1'] < 0.5 else
          '#e67e22' if e2e_pc.loc[c,'f1'] < 0.7 else '#27ae60'
          for c in e2e_pc.index]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(e2e_pc.index, e2e_pc['f1'], color=colors, edgecolor='white')
ax.axhline(e2e_pc['f1'].mean(), color='black', linestyle='--',
           label=f'Mean={e2e_pc["f1"].mean():.3f}')
ax.set_ylim(0, 1.1)
ax.set_title('End-to-End Per-Class F1 (7 classes, full test set)')
ax.legend()
for bar, v in zip(bars, e2e_pc['f1']):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.2f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()
display(e2e_pc)

## 7. Results Summary

In [ ]:
print('=' * 62)
print(f'  Run: {s1_result["run_name"]}')
print(f'  Dir: {RUN_DIR}')
print('=' * 62)

print(f'\n Stage 1 — Binary ({cfg["stage1"]["model"]["name"]})')
print(f'   Accuracy    : {s1_metrics["accuracy"]:.4f}')
print(f'   F1 (emotion): {s1_metrics["f1"]:.4f}')
print(f'   Macro F1    : {s1_metrics["macro_f1"]:.4f}')

print(f'\n Stage 2 — 6 Emotions ({cfg["stage2"]["model"]["name"]})')
print(f'   Micro F1    : {s2_metrics["micro_f1"]:.4f}')
print(f'   Macro F1    : {s2_metrics["macro_f1"]:.4f}')
print(f'   Subset Acc  : {s2_metrics["subset_accuracy"]:.4f}')

print(f'\n End-to-End — 7 Classes')
print(f'   Micro F1    : {e2e["micro_f1"]:.4f}')
print(f'   Macro F1    : {e2e["macro_f1"]:.4f}')
print(f'   Weighted F1 : {e2e["weighted_f1"]:.4f}')
print(f'   Subset Acc  : {e2e["subset_accuracy"]:.4f}')
print('=' * 62)

## 8. Inference on New Texts

In [ ]:
import torch
from src.train import build_model
from src.dataloader import BACKBONE_REGISTRY, EMOTION_NAMES
from src.utils import apply_threshold
from transformers import AutoTokenizer

s1_name = cfg['stage1']['model']['name']
s2_name = cfg['stage2']['model']['name']

s1_ckpt = torch.load(os.path.join(RUN_DIR, 'checkpoints', 'stage1_best.pth'),
                     map_location=device, weights_only=False)
s2_ckpt = torch.load(os.path.join(RUN_DIR, 'checkpoints', 'stage2_best.pth'),
                     map_location=device, weights_only=False)

s1_model = build_model(cfg['stage1'], stage='stage1').to(device)
s1_model.load_state_dict(s1_ckpt['model_state'])
s1_model.eval()

s2_model = build_model(cfg['stage2'], stage='stage2').to(device)
s2_model.load_state_dict(s2_ckpt['model_state'])
s2_model.eval()

s1_tok = AutoTokenizer.from_pretrained(BACKBONE_REGISTRY[s1_name]['pretrained'])
s2_tok = AutoTokenizer.from_pretrained(BACKBONE_REGISTRY[s2_name]['pretrained'])

s1_thr = e2e['best_threshold_s1']
s2_thr = e2e['best_thresholds_s2']

def predict(text):
    ml = cfg['data']['max_length']
    def enc(tok):
        e = tok(text, max_length=ml, padding='max_length', truncation=True, return_tensors='pt')
        return e['input_ids'].to(device), e['attention_mask'].to(device)
    with torch.no_grad():
        p1 = torch.sigmoid(s1_model(*enc(s1_tok))).item()
        if p1 < s1_thr:
            print(f'  [{text[:60]}]')
            print(f'  → NEUTRAL  (p_emotion={p1:.3f})')
            return
        p2 = torch.sigmoid(s2_model(*enc(s2_tok))).cpu().numpy()[0]
        preds = (p2 >= s2_thr).astype(int)
        detected = {n: float(p2[i]) for i,n in enumerate(EMOTION_NAMES) if preds[i]==1}
        if not detected:
            top = EMOTION_NAMES[int(p2.argmax())]
            detected = {top: float(p2.max())}
    print(f'  [{text[:60]}]')
    print(f'  → EMOTION  (p_emotion={p1:.3f})')
    for n, p in sorted(detected.items(), key=lambda x: -x[1]):
        print(f'      {n:<12}: {p:.3f}  {"█"*int(p*20)}')
    print()

tests = [
    "I am so incredibly happy today!",
    "I hate being lied to — it makes me so furious.",
    "The thunder scared me, I couldn't sleep.",
    "I just read an article about climate change.",
    "Wow, that plot twist I did NOT see coming!",
    "My grandmother passed away. I miss her so much.",
    "The report will be submitted by Friday.",
    "I'm disgusted by the corruption in the government.",
    "I'm scared AND angry about what happened.",
]
for t in tests:
    predict(t)